In [2]:
import pandas as pd
import requests
import pandas as pd
import json
import jsonschema
from jsonschema import validate
import re

In [3]:

from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../../.env") # goes up from Part 4 to root
api_key = os.getenv("OPENROUTER_API_KEY")

if api_key is None:
    raise ValueError("OPENROUTER_API_KEY not found. Please check your .env file.")

print("API Key loaded successfully:", api_key is not None)


API Key loaded successfully: True


In [4]:
url = "https://openrouter.ai/api/v1/chat/completions"

# LLM model to use
model = "openai/gpt-4.1-mini"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """
    Sends a request to the OpenRouter LLM API and returns the model response.
    """

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Request failed with status code:", response.status_code)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [5]:
response = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0
)

print(response)

hello


Task 2 - Prompt design 

In [6]:
df=pd.read_csv("../Data/cleaned_data.csv")
print("Loaded csv file successfully!")
df.columns.tolist()
# print(f"First 5 rows:\n{df.head(5)}")
# print(f"Datatypes:\n{df.dtypes}")
# print(f"Shape of Dataframe:\n{df.shape}")
# print(f"Null value counts:\n{df.isnull().sum()}")
# print(f"Null value percentage:\n{round((df.isnull().sum() / df.shape[0] )* 100,2)}")
sample_df = df.sample(3, random_state=42)

sample_df

Loaded csv file successfully!


,City,Date,season,AQI_Bucket,avg_temp,min_temp,max_temp,wind_speed,air_pressure,AQI,ONI
8759,Lucknow,2018-05-23,Summer,Poor,36.95,29.15,44.05,11.1,1009.9,257.0,-0.13
3921,Chennai,2019-11-23,Post-monsoon,Satisfactory,25.70,23.20,30.30,5.4,1013.6,64.0,0.58
3468,Chennai,2018-08-27,Monsoon,Satisfactory,29.50,26.00,35.70,11.2,1005.1,70.0,0.27


In [7]:

json_records = sample_df.to_dict(orient="records")
json_records

[{'City': 'Lucknow',
  'Date': '2018-05-23',
  'season': 'Summer',
  'AQI_Bucket': 'Poor',
  'avg_temp': 36.95,
  'min_temp': 29.15,
  'max_temp': 44.05,
  'wind_speed': 11.1,
  'air_pressure': 1009.9,
  'AQI': 257.0,
  'ONI': -0.13},
 {'City': 'Chennai',
  'Date': '2019-11-23',
  'season': 'Post-monsoon',
  'AQI_Bucket': 'Satisfactory',
  'avg_temp': 25.7,
  'min_temp': 23.2,
  'max_temp': 30.3,
  'wind_speed': 5.4,
  'air_pressure': 1013.6,
  'AQI': 64.0,
  'ONI': 0.58},
 {'City': 'Chennai',
  'Date': '2018-08-27',
  'season': 'Monsoon',
  'AQI_Bucket': 'Satisfactory',
  'avg_temp': 29.5,
  'min_temp': 26.0,
  'max_temp': 35.7,
  'wind_speed': 11.2,
  'air_pressure': 1005.1,
  'AQI': 70.0,
  'ONI': 0.27}]

Task 3 Structured output handling for Track A



In [8]:
system_prompt = """
You are a structured data extractor.

Convert AQI dataset records into the required JSON format.

Rules:
- Output only valid JSON.
- Do not add explanations.
- Do not use markdown.
- Keep values from the input record.

For the summary field, generate a short health-related statement based on AQI category:
- Good: Air quality is Good .
-Satisfactory: Air Quality is Satisfactory
- Moderate: Air quality is acceptable but sensitive groups should take precautions.
- Poor: High AQI may affect health and outdoor exposure should be reduced.
- Very Poor: Avoid prolonged outdoor activities due to unhealthy air quality.
- Severe: Limit outdoor exposure due to hazardous air quality.
"""

example1 = json_records[0]
example2 = json_records[1]
actual_input = json_records[2]

# print(example1)
# print(example2)
def create_user_prompt(actual_record):

    example1 = json_records[0]
    example2 = json_records[1]

    user_prompt = f"""

Example 1

Input:
{json.dumps(example1, indent=2)}

Output:
{{
 "city": "{example1['City']}",
 "aqi": {example1['AQI']},
 "category": "{example1['AQI_Bucket']}",
 "season": "{example1['season']}",
 "temperature": {example1['avg_temp']},
 "summary": "High AQI may affect health and outdoor exposure should be reduced."
}}


Example 2

Input:
{json.dumps(example2, indent=2)}

Output:
{{
 "city": "{example2['City']}",
 "aqi": {example2['AQI']},
 "category": "{example2['AQI_Bucket']}",
 "season": "{example2['season']}",
 "temperature": {example2['avg_temp']},
 "summary": "Air Quality is Satisfactory."
}}


Actual Input:

{json.dumps(actual_record, indent=2)}

Output:

"""

    return user_prompt


Tempearture 0 and  7

In [9]:
temperature_results = []

for i, record in enumerate(json_records):

    user_prompt = create_user_prompt(record)

    output_0 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    output_07 = call_llm(
        system_prompt,
        user_prompt,
        temperature=0.7
    )

    temperature_results.append({
        "Input": record,
        "Output_temp_0": output_0,
        "Output_temp_07": output_07
    })


In [10]:
temperature_results
# df=pd.read_csv("../Data/cleaned_data.csv")
# for item in temperature_results:
#     item['Output_temp_0'] = json.loads(item['Output_temp_0'])
#     item['Output_temp_07'] = json.loads(item['Output_temp_07'])




# with open("../Data/temperature_results.json", "w") as f:
#     json.dump(temperature_results, f, indent=2)

[{'Input': {'City': 'Lucknow',
   'Date': '2018-05-23',
   'season': 'Summer',
   'AQI_Bucket': 'Poor',
   'avg_temp': 36.95,
   'min_temp': 29.15,
   'max_temp': 44.05,
   'wind_speed': 11.1,
   'air_pressure': 1009.9,
   'AQI': 257.0,
   'ONI': -0.13},
  'Output_temp_0': '{\n "city": "Lucknow",\n "aqi": 257.0,\n "category": "Poor",\n "season": "Summer",\n "temperature": 36.95,\n "summary": "High AQI may affect health and outdoor exposure should be reduced."\n}',
  'Output_temp_07': '{\n  "city": "Lucknow",\n  "aqi": 257.0,\n  "category": "Poor",\n  "season": "Summer",\n  "temperature": 36.95,\n  "summary": "High AQI may affect health and outdoor exposure should be reduced."\n}'},
 {'Input': {'City': 'Chennai',
   'Date': '2019-11-23',
   'season': 'Post-monsoon',
   'AQI_Bucket': 'Satisfactory',
   'avg_temp': 25.7,
   'min_temp': 23.2,
   'max_temp': 30.3,
   'wind_speed': 5.4,
   'air_pressure': 1013.6,
   'AQI': 64.0,
   'ONI': 0.58},
  'Output_temp_0': '{\n  "city": "Chennai",\n 

In [11]:
output_schema = {
    "type": "object",
    "properties": {
        "city": {
            "type": "string"
        },
        "aqi": {
            "type": "number"
        },
        "category": {
            "type": "string"
        },
        "season": {
            "type": "string"
        },
        "temperature": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        }
    },
    "required": [
        "city",
        "aqi",
        "category",
        "season",
        "temperature",
        "summary"
    ]
}

In [12]:
def validate_llm_output(response):

    try:
        
        response = response.strip()

       
        parsed_op = json.loads(response)

    except json.JSONDecodeError as e:
        print("JSON parsing failed:")
        print(e)
        return None


    try:
        
        validate(
            instance=parsed_op,
            schema=output_schema
        )

        print("Validation successful")
        return parsed_op

    except jsonschema.ValidationError as e:
       print("Schema validation failed:", e.message)
       return {"city": None, "aqi": None, "category": "UNKNOWN", "season": None, "temperature": None, "summary": "validation_failed"}

In [13]:
validated_output = validate_llm_output(output_07)

validated_output

Validation successful


{'city': 'Chennai',
 'aqi': 70.0,
 'category': 'Satisfactory',
 'season': 'Monsoon',
 'temperature': 29.5,
 'summary': 'Air Quality is Satisfactory.'}

In [14]:
def run_extraction_pipeline(record):

    user_prompt = create_user_prompt(record)

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print("Raw LLM Response:")
    print(response)

    validated_result = validate_llm_output(response)

    return validated_result

In [15]:
validation_results = []

for record in json_records:

    
    print("Input Record:")
    print(record)

    result = run_extraction_pipeline(record)

    validation_results.append({
        "Input": record,
        "Output": result
    })

    print("Validation Result:")
    print(result)

Input Record:
{'City': 'Lucknow', 'Date': '2018-05-23', 'season': 'Summer', 'AQI_Bucket': 'Poor', 'avg_temp': 36.95, 'min_temp': 29.15, 'max_temp': 44.05, 'wind_speed': 11.1, 'air_pressure': 1009.9, 'AQI': 257.0, 'ONI': -0.13}
Raw LLM Response:
{
 "city": "Lucknow",
 "aqi": 257.0,
 "category": "Poor",
 "season": "Summer",
 "temperature": 36.95,
 "summary": "High AQI may affect health and outdoor exposure should be reduced."
}
Validation successful
Validation Result:
{'city': 'Lucknow', 'aqi': 257.0, 'category': 'Poor', 'season': 'Summer', 'temperature': 36.95, 'summary': 'High AQI may affect health and outdoor exposure should be reduced.'}
Input Record:
{'City': 'Chennai', 'Date': '2019-11-23', 'season': 'Post-monsoon', 'AQI_Bucket': 'Satisfactory', 'avg_temp': 25.7, 'min_temp': 23.2, 'max_temp': 30.3, 'wind_speed': 5.4, 'air_pressure': 1013.6, 'AQI': 64.0, 'ONI': 0.58}
Raw LLM Response:
{
 "city": "Chennai",
 "aqi": 64.0,
 "category": "Satisfactory",
 "season": "Post-monsoon",
 "tempe

In [16]:
validation_results

[{'Input': {'City': 'Lucknow',
   'Date': '2018-05-23',
   'season': 'Summer',
   'AQI_Bucket': 'Poor',
   'avg_temp': 36.95,
   'min_temp': 29.15,
   'max_temp': 44.05,
   'wind_speed': 11.1,
   'air_pressure': 1009.9,
   'AQI': 257.0,
   'ONI': -0.13},
  'Output': {'city': 'Lucknow',
   'aqi': 257.0,
   'category': 'Poor',
   'season': 'Summer',
   'temperature': 36.95,
   'summary': 'High AQI may affect health and outdoor exposure should be reduced.'}},
 {'Input': {'City': 'Chennai',
   'Date': '2019-11-23',
   'season': 'Post-monsoon',
   'AQI_Bucket': 'Satisfactory',
   'avg_temp': 25.7,
   'min_temp': 23.2,
   'max_temp': 30.3,
   'wind_speed': 5.4,
   'air_pressure': 1013.6,
   'AQI': 64.0,
   'ONI': 0.58},
  'Output': {'city': 'Chennai',
   'aqi': 64.0,
   'category': 'Satisfactory',
   'season': 'Post-monsoon',
   'temperature': 25.7,
   'summary': 'Air Quality is Satisfactory.'}},
 {'Input': {'City': 'Chennai',
   'Date': '2018-08-27',
   'season': 'Monsoon',
   'AQI_Bucket':

In [17]:
def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) 
        or 
        re.search(phone_pattern, text)
    )

In [18]:
def safe_call_llm(system_prompt, user_input, temperature=0):

    if has_pii(user_input):

        print("Input blocked: PII detected.")
        return None

    response = call_llm(
        system_prompt,
        user_input,
        temperature=temperature
    )

    return response

In [19]:
pii_input = """
Extract AQI information.
Contact person: divyaa@gmail.com
"""

result = safe_call_llm(
    system_prompt,
    pii_input,
    temperature=0
)

print(result)

Input blocked: PII detected.
None


In [20]:
safe_input = json.dumps(json_records[0], indent=2)

result = safe_call_llm(
    system_prompt,
    safe_input,
    temperature=0
)

print(result)

{
  "City": "Lucknow",
  "Date": "2018-05-23",
  "season": "Summer",
  "AQI_Bucket": "Poor",
  "avg_temp": 36.95,
  "min_temp": 29.15,
  "max_temp": 44.05,
  "wind_speed": 11.1,
  "air_pressure": 1009.9,
  "AQI": 257.0,
  "ONI": -0.13,
  "summary": "High AQI may affect health and outdoor exposure should be reduced."
}


In [ ]:
final_results = []

for record in json_records:

    
    print("Input:")
    print(record)


    
    user_prompt = create_user_prompt(record)


    
    llm_response = safe_call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )


    print("\nRaw LLm Repsonse:")
    print(llm_response)


   
    validation_result = validate_llm_output(llm_response)


    
    if validation_result is not None:
        print("Pass - Valid JSON")
        validation_status = "Pass"
    else:
        print("Fail - Invalid JSON")
        validation_status = "Fail"


    final_results.append({
        "Input": record,
        "LLM Output": validation_result,
        "Validation Status": validation_status,
        "Guardrail Result": "Pass - No PII detected"
    })

NameError: name 'json_records' is not defined